# Détecteur d'Émotions par Intelligence Artificielle

## Installation des Dépendances

In [ ]:
# Installez les packages si nécessaire
# !pip install tensorflow opencv-python pillow pandas numpy matplotlib seaborn -q

print("Installation vérifiée")

## Importation des Bibliothèques

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {len(tf.config.list_physical_devices('GPU'))} GPU")

## Configuration et Chemins

In [ ]:
# Chemins relatifs
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "Data"

# Paramètres du modèle
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
EPOCHS = 50  # Phase 1: 20 + Phase 2: 30 avec fine-tuning
VALIDATION_SPLIT = 0.2

# Classes d'émotions
CLASS_NAMES = ['Colère', 'Peur', 'Joie', 'Tristesse', 'Surprise']
NUM_CLASSES = len(CLASS_NAMES)

print(f"Répertoire de travail: {PROJECT_DIR}")
print(f"Données: {DATA_DIR}")
print(f"Taille d'image: {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"Nombre de classes: {NUM_CLASSES}")

## Chargement et Préparation des Données

In [ ]:
batch_size = 32
img_height = 64
img_width = 64

print("Chargement des données d'entraînement...")
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical'
)

print("\nChargement des données de validation...")
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical'
)

class_names = train_ds.class_names
num_classes = len(class_names)
print("\nClasses détectées :", class_names)

## Visualisation des Échantillons

In [ ]:
plt.figure(figsize=(15, 5))

for i, emotion in enumerate(class_names):
    folder_path = DATA_DIR / emotion
    if folder_path.exists():
        images = list(folder_path.glob('*'))
        if images:
            img_path = str(images[0])
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                plt.subplot(1, len(class_names), i + 1)
                plt.imshow(img)
                plt.title(f"{emotion}\n{img.shape[0]}x{img.shape[1]}", fontsize=11, fontweight='bold')
                plt.axis('off')

plt.tight_layout()
plt.show()
print("Échantillons d'images affichés")

## Analyse Exploratoire (EDA)

In [ ]:
# Analyse des classes
print("ANALYSE DES DONNÉES")
print("=" * 50)
print(f"\nNombre de classes: {len(class_names)}")
print(f"Classes détectées: {class_names}")

# Compter les images par classe
class_counts = {}
for emotion in class_names:
    folder_path = DATA_DIR / emotion
    if folder_path.exists():
        count = len(list(folder_path.glob('*')))
        class_counts[emotion] = count
        print(f"   • {emotion}: {count} images")

print(f"\nTotal d'images: {sum(class_counts.values())}")
print(f"Batch size: {batch_size}")
print(f"Taille entrée: {img_height}x{img_width}")

: 

## Modèle et Entraînement

In [ ]:
# Architecture simple et efficace
model = models.Sequential([
    layers.Input(shape=(img_height, img_width, 3)),
    layers.Rescaling(1./255),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Entraînement
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

## Sauvegarde du Modèle

In [ ]:
# Sauvegarde du modèle au format standard Keras
model_path = 'emotion_model.h5'
model.save(model_path)

print(f"Modèle sauvegardé sous : {model_path}")

# Sauvegarde des noms de classes
import pickle
with open('class_names.pkl', 'wb') as f:
    pickle.dump(class_names, f)

## Évaluation et Courbes d'apprentissage

In [ ]:
# Récupération des données de l'historique
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(14, 5))

# Graphique de l'Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Précision Entraînement')
plt.plot(epochs_range, val_acc, label='Précision Validation')
plt.legend(loc='lower right')
plt.title('Précision (Accuracy) - Entraînement vs Validation')

# Graphique de la Perte (Loss)
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Perte Entraînement')
plt.plot(epochs_range, val_loss, label='Perte Validation')
plt.legend(loc='upper right')
plt.title('Perte (Loss) - Entraînement vs Validation')

plt.show()

# Évaluation finale sur le set de validation
val_loss_score, val_accuracy = model.evaluate(val_ds)
print(f"\nPrécision finale sur le set de validation : {val_accuracy*100:.2f}%")